# CV — Deterministic Image Preprocessing (revised)

**Purpose:** establish a conservative, non-destructive preprocessing baseline for the SMC-LUD Clean patient-level split.

This notebook intentionally does **not** create a processed-image folder, report, or contact sheet. Images are transformed in memory and visualizations appear inline.

### Main decision

The previous square `Letterbox` preprocessing added approximately 19.8% black padding to Hemangioma images and 0% to HCC images. Since HCC images are all 512×512 while Hemangioma images are non-square, the added padding is almost a direct encoding of the class label.

The revised candidate therefore:

1. Applies EXIF orientation correction and RGB conversion.
2. Removes only complete near-black rows/columns at the image boundary (extremely conservative; not FOV segmentation).
3. Resizes the shorter side to 224 while preserving aspect ratio.
4. Adds **no padding**, performs **no center crop**, and does **not stretch** the image.
5. Uses ImageNet normalization because the first baseline uses ImageNet-pretrained ResNet18 weights.

The resulting tensors have variable spatial dimensions. ResNet18 supports variable H×W inputs through its adaptive global-average-pooling layer; training therefore uses batch size 1 with gradient accumulation.

**Important limitation:** this removes artificial black letterbox bands, but it does not erase the original HCC/Hemangioma aspect-ratio association. No safe image-only transform can guarantee removal of that association without potentially distorting or discarding anatomy. The remaining confounding must be reported, and independent acquisition-balanced data is ultimately needed.

ImageNet mean/std is kept here because this baseline uses ImageNet-pretrained ResNet18. The original SMC-LUD paper's train-set normalization is a different protocol and can be evaluated later as a separate, train-only normalization ablation; the two should not be mixed.

## Research basis and what we are/are not reproducing

- **SMC-LUD, Scientific Data (2026):** reports 224×224 resizing and training-set mean/standard-deviation normalization; uses the Clean version to avoid caliper overlays and a patient-level split. It does not document a detailed FOV-border-removal algorithm. [Original paper](https://www.nature.com/articles/s41597-026-07023-7).
- **SEED-Net, Frontiers in Medicine (2026):** reports EXIF correction, RGB conversion, removal of obvious outer borders/device-interface regions, preservation of the complete processed ultrasound field (lesion plus adjacent tissue), then aspect-ratio-preserving resize and padding to 224×224. It explicitly does not use lesion-centered or bounding-box crops. Its exact crop parameters are not fully specified, and its authors acknowledge that residual acquisition/layout/background shortcuts cannot be excluded. [Paper](https://doi.org/10.3389/fmed.2026.1942733).
- **Radiology shortcut-learning review (JACR, 2023):** describes spurious image markers and acquisition/context features as shortcut signals; it discusses data-centric mitigation and cross-population evaluation. [Review](https://doi.org/10.1016/j.jacr.2023.06.025).
- **DeGrave et al., Nature Machine Intelligence (2021):** showed that removing one visible cue by cropping does not necessarily remove shortcut learning; models can shift to other correlated image factors. This is why we avoid broad/automatic crops. [Paper](https://doi.org/10.1038/s42256-021-00338-7).

**Our implementation is deliberately more conservative than SEED-Net:** it removes only entire near-black edge strips. It does not attempt automatic sector/FOV segmentation or device-UI removal. Do not describe this as a replication of SEED-Net.

## 1. Environment and project paths

The Colab project layout is expected to be:

```text
/content/drive/MyDrive/LiverSira AI/Training/
├── data/
├── src/
└── notebooks/
```

The notebook changes directory to `Training/data` when running in Colab, then imports the existing project path configuration. It also supports local execution from a directory inside the project.

In [ ]:
import os
import sys
from pathlib import Path

# Colab layout requested by the project:
# /content/drive/MyDrive/LiverSira AI/Training/
# ├── data/
# ├── src/
# └── notebooks/
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LiverSira AI/Training")

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

if (COLAB_PROJECT_ROOT / "src" / "liversira_ai").is_dir():
    PROJECT_ROOT = COLAB_PROJECT_ROOT
    DATA_WORKDIR = PROJECT_ROOT / "data"
    if DATA_WORKDIR.is_dir():
        os.chdir(DATA_WORKDIR)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (current, *current.parents)
         if (p / "src" / "liversira_ai").is_dir()),
        None
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Project root not found. Expected Training/src/liversira_ai "
            "or /content/drive/MyDrive/LiverSira AI/Training/src/liversira_ai."
        )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from liversira_ai.common.paths import CV_PROCESSED_DIR

DATASET_ROOT = Path(CV_PROCESSED_DIR) / "patient_split_v1"
METADATA_PATH = DATASET_ROOT / "patient_split_metadata.csv"

print({
    "project_root": str(PROJECT_ROOT),
    "working_directory": str(Path.cwd()),
    "dataset_root": str(DATASET_ROOT),
    "metadata_exists": METADATA_PATH.is_file(),
})
if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Missing patient split metadata: {METADATA_PATH}")

## 2. Verify the patient-level split (Train and Validation only)

The physical split and manifest remain immutable. This notebook reads image paths only for Train and Validation. It does not open or inspect Test images.

Expected counts are Train = 1,770 (930 HCC, 840 Hemangioma) and Validation = 629 (332 HCC, 297 Hemangioma). Patient groups must not overlap between these two splits.

In [ ]:
import hashlib
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageOps
from collections import Counter

metadata_all = pd.read_csv(METADATA_PATH)
required_columns = {
    "split", "label", "patient_group_id", "destination_rel_path"
}
missing_columns = required_columns - set(metadata_all.columns)
if missing_columns:
    raise KeyError(f"Missing required metadata columns: {sorted(missing_columns)}")

# Exclude Test before constructing any Test file paths.
metadata = metadata_all[
    metadata_all["split"].isin(["train", "val"])
].copy()
metadata["path"] = metadata["destination_rel_path"].map(
    lambda value: DATASET_ROOT / Path(str(value).replace("\\", "/"))
)

expected = {
    ("train", "HCC"): 930,
    ("train", "Hemangioma"): 840,
    ("val", "HCC"): 332,
    ("val", "Hemangioma"): 297,
}
actual = metadata.groupby(["split", "label"]).size().to_dict()

for key, expected_count in expected.items():
    actual_count = int(actual.get(key, 0))
    assert actual_count == expected_count, (
        f"Count mismatch for {key}: {actual_count} != {expected_count}"
    )

assert len(metadata) == 2399
assert metadata["patient_group_id"].notna().all()
assert metadata["destination_rel_path"].is_unique

train_patients = set(
    metadata.loc[metadata["split"] == "train", "patient_group_id"]
)
val_patients = set(
    metadata.loc[metadata["split"] == "val", "patient_group_id"]
)
assert train_patients.isdisjoint(val_patients), "Train/Val patient overlap detected"

metadata["path_exists"] = metadata["path"].map(Path.is_file)
missing_paths = metadata.loc[~metadata["path_exists"], "path"].astype(str).tolist()
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths)} Train/Validation images are missing. "
        f"Examples: {missing_paths[:5]}"
    )

display(pd.crosstab(metadata["split"], metadata["label"], margins=True))
print({
    "train_images": int((metadata["split"] == "train").sum()),
    "validation_images": int((metadata["split"] == "val").sum()),
    "train_patients": len(train_patients),
    "validation_patients": len(val_patients),
    "train_val_patient_overlap": 0,
    "test_image_paths_constructed": False,
    "missing_train_val_files": 0,
})

# Optional manifest consistency check (metadata only; no Test image access).
try:
    from liversira_ai.common.paths import CV_MANIFEST_DIR
    manifest_path = Path(CV_MANIFEST_DIR) / "patient_level_manifest.csv"
    if manifest_path.is_file():
        manifest = pd.read_csv(manifest_path)
        if {"split", "label", "patient_group_id"}.issubset(manifest.columns):
            manifest_tv = manifest[manifest["split"].isin(["train", "val"])]
            manifest_counts = manifest_tv.groupby(["split", "label"]).size().to_dict()
            for key, expected_count in expected.items():
                assert int(manifest_counts.get(key, 0)) == expected_count
            print("Train/Validation allocation agrees with patient_level_manifest.csv.")
        else:
            print("Manifest found, but its columns differ; sidecar counts remain authoritative.")
    else:
        print(f"Manifest not found at {manifest_path}; using physical split sidecar.")
except ImportError:
    print("CV_MANIFEST_DIR unavailable; using physical split sidecar.")

## 3. Evidence carried forward from the previous audits

These are **previously measured findings**, not recalculated in this notebook:

| Finding on this project split | Observed result |
|---|---:|
| Train HCC square images | 930 / 930 |
| Train Hemangioma square images | 0 / 840 |
| Patient-level 5-fold AUC: aspect ratio | 1.0000 |
| Patient-level 5-fold AUC: black letterbox padding ratio | 1.0000 |
| Patient-level 5-fold AUC: black pixel ratio | 0.7973 |
| Patient-level 5-fold AUC: dark pixel ratio | 0.7583 |
| Patient-level 5-fold AUC: mean intensity | 0.6888 |
| Train JPEG: HCC / Hemangioma | 399 / 840 |
| Train JPEG mean quantization-table statistic | 7.242 / 36.234 |

Aspect ratio and letterbox-padding ratio are mathematically linked for square output canvases; they are not independent pieces of evidence. The near-perfect aspect-ratio result is the key warning. The JPEG statistic is descriptive only; it is not a calibrated JPEG-quality score and does not prove the model uses compression artifacts.

## 4. Deterministic preprocessing functions

The trim rule only removes an outer row/column when at least 99.9% of its pixels are at or below intensity 3 (grayscale). Each edge is capped at 2% of the corresponding image dimension. A fan-shaped black corner is not a complete row or column and is therefore retained.

This is a conservative **solid-frame trim**, not a method for segmenting the ultrasound fan or detecting the lesion.

In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import torch

# Conservative, deterministic policy:
# - apply EXIF orientation;
# - convert to RGB;
# - remove only complete near-black outer rows/columns;
# - resize the shorter side to 224 while preserving aspect ratio;
# - do not center-crop, stretch, or add letterbox padding.
SHORT_SIDE = 224
DARK_THRESHOLD = 3
DARK_EDGE_FRACTION = 0.999
MAX_TRIM_FRACTION_PER_EDGE = 0.02

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def trim_uniform_black_border(
    image,
    threshold=DARK_THRESHOLD,
    dark_fraction=DARK_EDGE_FRACTION,
    max_fraction=MAX_TRIM_FRACTION_PER_EDGE,
):
    """
    Remove only complete near-black strips touching the outer image frame.

    A row/column is removed only when at least 99.9% of its pixels are
    <= threshold. Fan-shaped black corners are not removed because they
    do not occupy a complete row/column. No lesion or FOV segmentation
    is attempted.
    """
    image = ImageOps.exif_transpose(image).convert("RGB")
    gray = np.asarray(image.convert("L"), dtype=np.uint8)
    h, w = gray.shape

    row_dark_fraction = (gray <= threshold).mean(axis=1)
    col_dark_fraction = (gray <= threshold).mean(axis=0)

    max_top = int(h * max_fraction)
    max_bottom = int(h * max_fraction)
    max_left = int(w * max_fraction)
    max_right = int(w * max_fraction)

    top = 0
    while top < max_top and row_dark_fraction[top] >= dark_fraction:
        top += 1

    bottom = 0
    while bottom < max_bottom and row_dark_fraction[h - 1 - bottom] >= dark_fraction:
        bottom += 1

    left = 0
    while left < max_left and col_dark_fraction[left] >= dark_fraction:
        left += 1

    right = 0
    while right < max_right and col_dark_fraction[w - 1 - right] >= dark_fraction:
        right += 1

    # Guard against accidentally collapsing an image.
    if h - top - bottom < 32 or w - left - right < 32:
        top = bottom = left = right = 0

    cropped = image.crop((left, top, w - right, h - bottom))

    info = {
        "original_width": w,
        "original_height": h,
        "trim_top": top,
        "trim_bottom": bottom,
        "trim_left": left,
        "trim_right": right,
        "trimmed_width": cropped.width,
        "trimmed_height": cropped.height,
        "trimmed_fraction": 1.0 - (
            cropped.width * cropped.height / (w * h)
        ),
        "aspect_ratio_before": w / h,
        "aspect_ratio_after": cropped.width / cropped.height,
        "padding_pixels": 0,
    }
    return cropped, info


def resize_short_side_no_pad(image, short_side=SHORT_SIDE):
    """Resize while preserving aspect ratio; never crop or pad."""
    w, h = image.size
    scale = short_side / min(w, h)
    new_w = max(1, round(w * scale))
    new_h = max(1, round(h * scale))
    return image.resize((new_w, new_h), resample=Image.Resampling.BILINEAR)


def preprocess_image(image_or_path, short_side=SHORT_SIDE):
    """The single deterministic preprocessing function used in both notebooks."""
    if isinstance(image_or_path, (str, Path)):
        with Image.open(image_or_path) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
            image.load()
    else:
        image = ImageOps.exif_transpose(image_or_path).convert("RGB")

    trimmed, info = trim_uniform_black_border(image)
    resized = resize_short_side_no_pad(trimmed, short_side=short_side)

    info.update({
        "output_width": resized.width,
        "output_height": resized.height,
        "output_aspect_ratio": resized.width / resized.height,
        "output_short_side": min(resized.size),
    })
    return resized, info


def image_to_normalized_tensor(image):
    """RGB PIL image -> float32 CHW tensor with ImageNet normalization."""
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
    mean = torch.tensor(IMAGENET_MEAN, dtype=tensor.dtype).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=tensor.dtype).view(3, 1, 1)
    tensor = (tensor - mean) / std
    if not torch.isfinite(tensor).all():
        raise ValueError("Preprocessing generated NaN or Inf values.")
    return tensor


def legacy_black_letterbox(image, target=224):
    """Reference-only display of the old black-letterbox transform."""
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    scale = min(target / w, target / h)
    resized = image.resize(
        (max(1, round(w * scale)), max(1, round(h * scale))),
        resample=Image.Resampling.BILINEAR,
    )
    canvas = Image.new("RGB", (target, target), (0, 0, 0))
    x = (target - resized.width) // 2
    y = (target - resized.height) // 2
    canvas.paste(resized, (x, y))
    content_ratio = (resized.width * resized.height) / (target * target)
    return canvas, 1.0 - content_ratio


## 5. Inline comparison: old Letterbox vs revised no-padding input

Review samples from different Train patients. The revised image should retain its geometry and all ultrasound content while avoiding newly-added black canvas bands. The original image is never modified.

In [ ]:
# Show representative images from distinct Train patients.
rng = np.random.default_rng(42)
sample_rows = []

train_meta = metadata[metadata["split"] == "train"].copy()
for label in ["HCC", "Hemangioma"]:
    candidates = (
        train_meta[train_meta["label"] == label]
        .drop_duplicates("patient_group_id")
        .reset_index(drop=True)
    )
    n = min(3, len(candidates))
    chosen = rng.choice(len(candidates), size=n, replace=False)
    sample_rows.extend(candidates.iloc[chosen].to_dict("records"))

summary_rows = []
fig, axes = plt.subplots(len(sample_rows), 3, figsize=(14, 4 * len(sample_rows)))
axes = np.atleast_2d(axes)

for row_axes, record in zip(axes, sample_rows):
    with Image.open(record["path"]) as opened:
        original = ImageOps.exif_transpose(opened).convert("RGB")

    legacy, legacy_padding = legacy_black_letterbox(original, target=224)
    processed, info = preprocess_image(original, short_side=224)

    row_axes[0].imshow(original)
    row_axes[0].set_title(
        f"Original — {record['label']} / {record['patient_group_id']}\n"
        f"{original.width}×{original.height}"
    )

    row_axes[1].imshow(legacy)
    row_axes[1].set_title(
        f"Previous black Letterbox 224\nAdded padding: {legacy_padding:.1%}"
    )

    row_axes[2].imshow(processed)
    row_axes[2].set_title(
        f"Revised: no pad, no stretch, no center crop\n"
        f"{processed.width}×{processed.height} | "
        f"trimmed {info['trimmed_fraction']:.3%}"
    )

    for axis in row_axes:
        axis.axis("off")

    summary_rows.append({
        "label": record["label"],
        "patient_group_id": record["patient_group_id"],
        "original_size": original.size,
        "output_size": processed.size,
        "aspect_ratio_before": info["aspect_ratio_before"],
        "aspect_ratio_after": info["aspect_ratio_after"],
        "trimmed_fraction": info["trimmed_fraction"],
        "legacy_added_black_padding": legacy_padding,
        "new_added_padding": 0.0,
    })

plt.tight_layout()
plt.show()
display(pd.DataFrame(summary_rows).round(4))

## 6. Tensor and shape validation

The output short side is 224, but the long side varies by image. This is intentional: the model receives the complete image field without square padding. ImageNet normalization is applied channel-wise after conversion to RGB float values in [0, 1].

In [ ]:
# Tensor sanity check on a representative Train image.
sample_path = sample_rows[0]["path"]
sample_image, sample_info = preprocess_image(sample_path, short_side=SHORT_SIDE)
sample_tensor = image_to_normalized_tensor(sample_image)

print({
    "tensor_shape_CHW": tuple(sample_tensor.shape),
    "dtype": str(sample_tensor.dtype),
    "finite": bool(torch.isfinite(sample_tensor).all()),
    "min_after_normalization": float(sample_tensor.min()),
    "max_after_normalization": float(sample_tensor.max()),
    "output_size_WH": sample_image.size,
    "short_side": min(sample_image.size),
    "padding_pixels": sample_info["padding_pixels"],
    "normalization": "ImageNet mean/std (ResNet18 pretrained weights)",
})
assert sample_tensor.ndim == 3 and sample_tensor.shape[0] == 3
assert min(sample_tensor.shape[1:]) == SHORT_SIDE
assert sample_info["padding_pixels"] == 0

## 7. Decision and limitations

**Adopt this as the next experimental preprocessing baseline, not as a claim that all shortcuts are solved.**

- Safe deterministic edge strips may be removed.
- Fan-shaped regions, acoustic shadows, anechoic regions, lesions, and surrounding liver tissue are retained.
- No direct resize, center crop, sector segmentation, JPEG re-encoding, CLAHE, histogram equalization, denoising, or augmentation is applied.
- No Test images are inspected.
- The Aspect-Ratio shortcut remains a known confounder. The no-padding input removes the *added padding cue*, but not the underlying source geometry.
- A reliable solution to the remaining geometry/format confounding may require acquisition-balanced images or external data with overlapping image geometries across classes. Do not force aspect ratios to match by warping anatomy.

The next notebook trains a ResNet18 feature-extractor baseline using this exact transform, variable-size inputs, patient-balanced image weights, and both image-level and patient-level Validation metrics.